<a href="https://colab.research.google.com/github/Raksh575/Automation-Testing/blob/main/Agri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from sklearn.cluster import KMeans
import folium
import time

# Load your dataset (this should be replaced with your actual data loading code)
# For demonstration, I'll create a sample DataFrame structure
'''data = {
    'Crop': ['Pepper', 'Pepper', 'Banana', 'Banana', 'Tea', 'Tea'],
    'District': ['Namakkal', 'Dindigul', 'Erode', 'Thoothukudi', 'Nilgiris', 'Coimbatore'],
    'Taluk': ['Rasipuram', 'Kodaikanal', 'Modakurichi', 'Srivaikuntam', 'Coonoor', 'Valparai'],
    'Area (ha)': [800, 600, 5000, 2800, 20000, 6500]  # Numeric values without commas
}'''

df = pd.read_excel('/content/Dataset.xlsx')

# Data cleaning - ensure Area (ha) is numeric
try:
    # First try direct conversion (if values are already numeric)
    df['Area (ha)'] = pd.to_numeric(df['Area (ha)'], errors='raise')
except ValueError:
    # If that fails, try cleaning string values
    df['Area (ha)'] = pd.to_numeric(df['Area (ha)'].astype(str).str.replace(',', ''), errors='coerce')

# Drop rows with missing/invalid area values
df = df.dropna(subset=['Area (ha)'])

# Geocoding with caching and fallback
geolocator = Nominatim(user_agent="tn_agri_office_v3")

# District center coordinates as fallback
district_coords = {
    'Salem': (11.6500, 78.1333),
    'Theni': (10.0104, 77.4768),
    'Tiruchirappalli': (10.7833, 78.6833),
    'Dharmapuri': (12.1065, 78.1361),
    'Kallakurichi': (11.7380, 78.9620),
    'Tiruppur': (11.1107, 77.3480),
    'Thanjavur': (10.7870, 79.1378),
    'Kanniyakumari': (8.0883, 77.5385),
    'Tiruvannamalai': (12.2286, 79.0665),
    'Cuddalore': (11.7200, 79.7700),  # Adjusted to more precise coordinates
    'Villupuram': (11.9400, 79.5000),  # Adjusted to more precise coordinates
    # Previously provided districts:
    'Namakkal': (11.2186, 78.1682),
    'Dindigul': (10.3650, 77.9800),
    'Erode': (11.3428, 77.7274),
    'Thoothukudi': (8.7642, 78.1348),
    'Nilgiris': (11.4000, 76.7000),
    'Coimbatore': (11.0168, 76.9558)
}

def get_coordinates(district, taluk):
    cache_key = f"{taluk}, {district}"
    try:
        # Try with taluk + district
        location = geolocator.geocode(f"{taluk}, {district}, Tamil Nadu, India")
        if location:
            return (location.latitude, location.longitude)

        # Fallback to district center
        return district_coords.get(district, (None, None))
    except Exception as e:
        print(f"Geocoding failed for {cache_key}: {str(e)}")
        return district_coords.get(district, (None, None))
    finally:
        time.sleep(1)  # Rate limiting

# Add coordinates - in production you should cache these results
print("Geocoding locations...")
df['Coordinates'] = df.apply(lambda x: get_coordinates(x['District'], x['Taluk']), axis=1)
df = df.dropna(subset=['Coordinates'])
df[['Latitude', 'Longitude']] = pd.DataFrame(df['Coordinates'].tolist(), index=df.index)

def calculate_weighted_centroid(crop_df):
    valid_rows = crop_df.dropna(subset=['Latitude', 'Longitude', 'Area (ha)'])
    if len(valid_rows) == 0:
        return None

    total_area = valid_rows['Area (ha)'].sum()
    if total_area == 0:
        return None

    weighted_lat = np.sum(valid_rows['Latitude'] * valid_rows['Area (ha)']) / total_area
    weighted_lon = np.sum(valid_rows['Longitude'] * valid_rows['Area (ha)']) / total_area

    # Find nearest actual location
    def distance(row):
        return geodesic((weighted_lat, weighted_lon), (row['Latitude'], row['Longitude'])).km

    valid_rows['distance'] = valid_rows.apply(distance, axis=1)
    nearest = valid_rows.loc[valid_rows['distance'].idxmin()]

    return {
        'crop': nearest['Crop'],
        'centroid_lat': weighted_lat,
        'centroid_lon': weighted_lon,
        'recommended_district': nearest['District'],
        'recommended_taluk': nearest['Taluk'],
        'total_area': total_area,
        'num_locations': len(valid_rows)
    }

# Calculate centroids for all crops
crop_centroids = {}
for crop in df['Crop'].unique():
    crop_df = df[df['Crop'] == crop]
    centroid = calculate_weighted_centroid(crop_df)
    if centroid:
        crop_centroids[crop] = centroid

def optimize_agri_offices(crop_name, max_offices=3):
    if crop_name not in crop_centroids:
        return {"error": f"Crop '{crop_name}' not found in database"}

    crop_df = df[df['Crop'] == crop_name].dropna(subset=['Latitude', 'Longitude', 'Area (ha)'])

    # For small number of locations, recommend the top by area
    if len(crop_df) <= 2:
        best = crop_df.sort_values('Area (ha)', ascending=False).iloc[0]
        return {
            "recommendations": [{
                "district": best['District'],
                "taluk": best['Taluk'],
                "latitude": best['Latitude'],
                "longitude": best['Longitude'],
                "area_covered": best['Area (ha)'],
                "coverage_percentage": 100
            }]
        }

    # Weighted clustering
    X = crop_df[['Latitude', 'Longitude']].values
    weights = crop_df['Area (ha)'].values

    # Determine optimal number of offices (1-3)
    n_clusters = min(max_offices, max(1, len(crop_df) // 5))

    # Weighted sampling for clustering
    weighted_samples = np.repeat(X, weights.astype(int), axis=0)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(weighted_samples)

    recommendations = []
    for center in kmeans.cluster_centers_:
        # Find locations in this cluster
        distances = np.array([geodesic(center, x).km for x in X])
        cluster_mask = distances < np.percentile(distances, 75)

        if not cluster_mask.any():
            continue

        # Select the location with maximum area in cluster
        best_in_cluster = crop_df[cluster_mask].sort_values('Area (ha)', ascending=False).iloc[0]
        cluster_area = crop_df[cluster_mask]['Area (ha)'].sum()

        recommendations.append({
            "district": best_in_cluster['District'],
            "taluk": best_in_cluster['Taluk'],
            "latitude": float(center[0]),
            "longitude": float(center[1]),
            "area_covered": float(cluster_area),
            "coverage_percentage": round(100 * cluster_area / crop_df['Area (ha)'].sum(), 2)
        })

    return {"recommendations": sorted(recommendations, key=lambda x: -x['area_covered'])}

# Example usage
print("Banana offices:")
print(optimize_agri_offices("Banana"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/Dataset.xlsx'